<!-- cabecera-entorno -->
## Antes de empezar

**Clase 8 · Principios de visualización** — Bloque 2 · Demo. Este cuaderno se recorre **por su
cuenta**: explica cada concepto antes de usarlo, y el profesor circula por el salón resolviendo
dudas. No hay que esperar a que alguien lo dicte desde el tablero.

**Aquí no hay nada que teclear.** Todo el código está escrito y ejecutable, incluidas las figuras.
Usted lo corre, mira la salida y lee la explicación que está justo encima. Lo que sí le toca son
las **doce preguntas de interpretación**. Escribir código es el bloque 3, y es lo que se entrega.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `demo.ipynb` como
`demo_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md).

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| `FileNotFoundError` al leer el CSV | El notebook se abrió desde otra carpeta, o falta hacer `git pull` | Manual, problema 6 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
import sys
from pathlib import Path

try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Falta la librería '{error.name}'. Active el entorno virtual y seleccione el intérprete "
        ".venv en VSCode (Ctrl+Shift+P > Python: Select Interpreter), luego reinicie el kernel. "
        "Ver ../INSTALACION.md, problema 5."
    ) from error

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")

RUTA_VERIFICACION = "../datos/EJECUCION_PRESUPUESTAL.csv"
if Path(RUTA_VERIFICACION).exists():
    print("Datos: encontrados en", RUTA_VERIFICACION)
else:
    print("FALTA el archivo", RUTA_VERIFICACION, "- abra en VSCode la carpeta raíz del curso",
          "y ejecute 'git pull'. Ver ../INSTALACION.md, problema 6.")

# Clase 8 · Demo — Principios de visualización

**Dataset:** `../datos/EJECUCION_PRESUPUESTAL.csv`, ejecución del Presupuesto General de la
Nación, descargado de datos.gov.co. 3.645 filas x 21 columnas.

## Cómo se usa este cuaderno

Está escrito para que usted avance solo. Cada bloque de código viene precedido de la explicación del
concepto que usa, y cada término nuevo se define la primera vez que aparece. **Hay que leer antes de
ejecutar.**

**El recorrido:**

| Sección | De qué va | Qué se lleva |
|---------|-----------|--------------|
| 0 | El dataset y las cuatro columnas de plata | Saber qué significan los números que va a graficar |
| 1 | Los nombres de columna, y tres errores provocados a propósito | Reconocer el error antes de que le pase |
| 2 | La limpieza mínima para poder graficar | Un DataFrame del que se puede fiar |
| 3 | Escalas y magnitudes: por qué el eje dice `1e13` | El arreglo de una línea que separa un gráfico legible de uno que no |
| 4 | La paleta del día y la función que limpia los ejes | Las herramientas que va a usar hoy y el resto del semestre |
| 5 | Rediseño 1: el pie chart de 32 sectores | Tipo de gráfico equivocado para la pregunta |
| 6 | Rediseño 2: barras arcoíris con eje truncado | El eje que miente sobre la magnitud |
| 7 | Rediseño 3: líneas sobre categorías | El gráfico que se lee bien y afirma algo falso |
| 8 | La checklist aplicada y el cierre | El criterio con el que se califica el reto |

**Aquí no hay nada que teclear.** Todo el código está escrito y ejecutable: usted lo corre, mira la
salida y lee la explicación que está justo encima. Las tres figuras buenas vienen escritas con su
título, sus dos ejes etiquetados y su fuente, y son **el modelo que el reto sí le va a cobrar**.
Escribir código es el bloque 3, y es lo que se entrega.

**Las doce preguntas de interpretación** son lo que sí le toca a usted. No llevan código: se
responden escribiendo en español en la celda, debajo de *Tu respuesta:*. Cada una trae un bloque
plegable *"Comparar con la respuesta esperada"*. **Escriba la suya primero y ábralo después.**
Abrirlo antes no le ahorra nada: lo que se evalúa en el reto y en la sustentación no es que sepa
teclear `ax.barh`, es que sepa mirar una figura y decir por qué está bien o por qué miente.

**Las secciones "Para entender qué está pasando"** explican el fundamento: por qué un gráfico miente,
qué es el data-ink ratio, por qué truncar un eje de barras es una afirmación falsa y no un detalle de
estilo. Son las que convierten esto en criterio en vez de recetas. Si ya las tiene claras, se pueden
saltar sin perder el hilo del código; si no, son la parte que más le va a servir en los momentos
evaluativos, donde lo que se mira es por qué eligió ese gráfico.

### Lo que este cuaderno da por sabido

No se repite aquí nada de lo que ya se explicó. Si algo de esta lista no le suena, vuelva al cuaderno
que lo enseña **antes** de seguir.

| Lo que se da por sabido | Dónde se explicó |
|-------------------------|------------------|
| Librería, alias, DataFrame, Series, índice, dtype | Clase 2, demo, secciones 1 a 5 |
| `read_csv`, `head`, `info`, `describe`, `shape` | Clase 2, demo, secciones 2 y 3 |
| Máscara booleana, `&`, `\|`, `~`, `.isin()` | Clase 2, demo, secciones 7 a 11 |
| Nulos, `NaN`, valores inválidos de dominio, `df.copy()` | Clase 3, demo, pasos 1 a 6 |
| `groupby` y agregaciones, la analogía de los M&Ms | Clase 4, demo, sección 4 |
| Histograma, boxplot, outliers | Clase 4, demo, secciones 5 y 6 |
| Scatter, correlación, heatmap, elegir gráfico según los tipos | Clase 5, demo, secciones 2 a 5 |

**Lo nuevo de hoy no es una función de pandas.** Es el criterio: cuándo un gráfico es el correcto,
cuándo miente, y qué se borra. La sintaxis de matplotlib que aparece hoy es corta y se explica
completa; lo que hay que llevarse son las decisiones, no los nombres de los parámetros.

### La checklist de diez puntos

Es la del bloque 1, y es con la que se califica el reto. Cada rediseño de hoy se cierra pasándola.

| # | Criterio |
|---|----------|
| 1 | ¿Pasa la prueba de los 5 segundos con alguien ajeno al equipo? |
| 2 | ¿El tipo de gráfico corresponde a la pregunta? |
| 3 | ¿Las categorías están ordenadas por valor, no alfabéticamente? |
| 4 | ¿El eje empieza en cero, o hay una razón declarada para que no? |
| 5 | ¿Borré todo lo que se podía borrar sin perder información? |
| 6 | ¿Hay un solo elemento resaltado y el resto en gris? |
| 7 | ¿Funciona en escala de grises? |
| 8 | ¿El título dice la conclusión y es verdadero? |
| 9 | ¿Los ejes tienen unidades? |
| 10 | ¿Está la fuente del dato? |

### Qué puede revisar una máquina y qué no

Este cuaderno no trae verificador, y conviene decir por qué: **no hay una única respuesta correcta
para un gráfico.** Lo que sí es verificable de una figura —que esté dibujada, que tenga título, que
los ejes estén etiquetados, que el eje arranque en cero, que exista la línea de referencia— lo
revisa el verificador del **reto**, porque ahí las figuras las escribe usted.

Los puntos 1, 7 y 8 de la checklist —los cinco segundos, la escala de grises y que el título sea
**verdadero**— no los puede revisar ningún programa, ni hoy ni en el reto. Esos los juzga usted, y
son exactamente los que se preguntan en la sustentación.

---

## 0. El dataset, antes de tocarlo

Regla de la casa: nunca se ejecuta una línea de código sobre un dataset que no se sabe qué es.

| Campo | Valor |
|-------|-------|
| Qué mide | Ejecución del Presupuesto General de la Nación |
| Origen | datos.gov.co |
| Tamaño | 3.645 filas x 21 columnas |
| Granularidad | Una fila = un rubro presupuestal de una entidad, con su fuente de financiación |
| Cobertura | 32 sectores, 221 entidades |

**Es una foto, no una película.** Muestra el estado de ejecución en un momento dado. No hay ninguna
columna de tiempo: ni año, ni mes, ni fecha. Esa frase parece un detalle y no lo es: vuelve a
aparecer, convertida en el error más grave del día, en el rediseño 3.

### Las cuatro columnas de plata

Sin entender esto los gráficos no significan nada. El presupuesto público funciona como un embudo:

| Columna | Qué es |
|---------|--------|
| `Apropiación Vigente` | Lo que le autorizaron gastar. El techo |
| `Compromisos` | Lo que ya comprometió en contratos firmados |
| `Obligaciones` | Lo que ya recibió y debe pagar |
| `Pagos` | Lo que efectivamente salió |

La relación es de embudo: `Apropiación >= Compromisos >= Obligaciones >= Pagos`.

**El porcentaje de ejecución** es la métrica que se usa todo el día, y es un cociente:

```
% Ejecución = Compromisos / Apropiación Vigente * 100
```

Un sector con mucha apropiación y poca ejecución tiene plata autorizada que no ha comprometido. Ese
cruce —mucho presupuesto, poca ejecución— es la pregunta más interesante que permite este archivo, y
es la figura 3 del reto.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 200)

# Este cuaderno vive en clase08/demo/, y el CSV dos carpetas más arriba, en datasets/
df = pd.read_csv('../datos/EJECUCION_PRESUPUESTAL.csv')

print('Filas:', df.shape[0])
print('Columnas:', df.shape[1])
df.head(3)

### Primer vistazo, antes de graficar nada

Cuatro preguntas se responden siempre en el mismo orden, y con la misma media docena de funciones.
Son las de la clase 2 y la clase 3, y no se saltan porque el tema de hoy sea visualización: **una
figura hereda todos los defectos de la tabla que la alimenta**.

| Pregunta | Función |
|----------|---------|
| ¿Cuántas filas y cuántas columnas hay? | `df.shape` |
| ¿De qué tipo es cada columna, y cuántos nulos tiene? | `df.info()` |
| ¿Los números están guardados como números, o como texto? | `df.dtypes` |
| ¿Dónde hay nulos y cuántos? | `df.isna().sum()` |
| ¿Cómo se reparten los valores de las columnas de plata? | `df.describe()` |
| ¿Cuántas categorías distintas hay? | `df[col].nunique()` |

In [ ]:
# ¿De qué tipo es cada columna, cuántos no-nulos tiene y cuánta memoria ocupa la tabla?
df.info()

In [ ]:
# ¿Los números están guardados como números? dtypes lo dice columna por columna.
# object = texto. Una columna de plata en object no se puede sumar ni graficar.
print(df.dtypes)

In [ ]:
# ¿Dónde hay nulos y cuántos? Se muestran solo las columnas que tienen al menos uno.
nulos = df.isna().sum()
print(nulos[nulos > 0] if (nulos > 0).any() else 'Ninguna columna tiene nulos.')

In [ ]:
# ¿Cómo se reparten las cuatro columnas de plata? describe da conteo, media, desviación,
# mínimo, cuartiles y máximo. Con .T (transpuesta) se lee una columna por fila.
columnas_plata = ['Apropiación Vigente', 'Compromisos', 'Obligaciones', 'Pagos']
df[columnas_plata].describe().T

In [ ]:
# ¿Cuántas categorías distintas hay en las columnas que van a ir al eje de un gráfico?
# nunique cuenta valores distintos: es lo que decide cuántas barras tendría la figura.
for columna in ['Nombre Sector', 'Nombre Entidad', 'Nombre Nivel Uno Rubro']:
    print(f'{columna}: {df[columna].nunique()} valores distintos')

**Lo que ya se ve.** Las cuatro columnas de plata son numéricas, así que se pueden sumar y
graficar sin convertir nada. `Nombre Sector` tiene 32 valores distintos y `Nombre Entidad` 221: la
primera cabe en un gráfico de barras, la segunda no cabe en ninguno. **Ese número decide la figura
antes que el gusto.** Y en `describe()` la distancia entre la mediana y el máximo anticipa lo que
pasa después: unos pocos rubros enormes al lado de miles de rubros pequeños.

---

## 1. Los nombres de columna, y tres errores provocados

**El concepto.** Los nombres de columna son **datos**, no deseos. Vienen como los escribió quien
publicó el archivo, con sus erratas y sus caracteres raros, y pandas los compara letra por letra. Un
nombre casi correcto no es un nombre correcto.

Los tres errores de abajo están provocados a propósito con `try / except`. **No son accidentes del
cuaderno: son contenido.** La idea es que vea el mensaje de error hoy, con calma, y no por primera
vez en el minuto 40 del reto.

`try / except` ejecuta el bloque de arriba y, si revienta, en lugar de detener el cuaderno entra al
bloque de abajo con el error capturado en la variable `error`. Es la forma de mostrar un error sin
que el cuaderno se pare.

Primero, los nombres reales. `repr()` muestra la cadena **tal cual**, con las comillas y los
caracteres invisibles a la vista.

In [ ]:
for i, c in enumerate(df.columns):
    print(i, '|', repr(c))

### Error 1 · La columna que dice "Niel"

Mire la columna número 10 de la lista de arriba. Dice `Nombre Niel Dos Rubro`. **Dice "Niel", no
"Nivel".** Es una errata de la fuente original.

**Y no se corrige.** Si usted la renombra en su copia, su código deja de funcionar con el archivo que
descargó su compañero, y con el que descargue usted mismo el mes que viene. Se escribe como está.

In [ ]:
try:
    df['Nombre Nivel Dos Rubro']          # el nombre "correcto", que no existe
except KeyError as error:
    print('KeyError:', error)
    print()
    print('Lo que hay que escribir es exactamente esto:', repr(df.columns[10]))

### Error 2 · El apóstrofo que no es un apóstrofo

La columna 2 se llama `Recursos´Presupuestales`. Ese carácter entre las dos palabras **no es un
apóstrofo**: es un acento agudo suelto (`´`, U+00B4). Y no hay espacio a los lados.

Escribirlo a mano es casi imposible. La forma profesional de acceder a una columna con nombre raro es
**copiarla desde `df.columns`**, no teclearla.

In [ ]:
try:
    df["Recursos'Presupuestales"]         # con apóstrofo normal: no existe
except KeyError:
    print('Con apóstrofo normal: KeyError.')

nombre_real = df.columns[2]
print('Copiado desde df.columns:', repr(nombre_real))
print('Valores distintos:', df[nombre_real].nunique())

### Error 3 · El que **no** lanza ningún error, y por eso es el peor

Los dos anteriores revientan y le avisan. Este no.

En la columna `Recursos´Presupuestales` hay dos valores que **se ven idénticos** en pantalla y que
pandas trata como categorías distintas. La celda de abajo los muestra dos veces: primero como se ven,
después con `repr()`.

In [ ]:
conteo = df[nombre_real].value_counts()
print('Como se ven:')
print(conteo[conteo.index.str.contains('PARAFIS')].to_string())

print()
print('Como son de verdad:')
for valor in df[nombre_real].unique():
    if 'PARAFIS' in valor:
        print('  ', repr(valor))

**El diagnóstico.** `RENTAS\xa0PARAFISCALES` y `RENTAS PARAFISCALES`.

El `\xa0` es un **espacio no separable** (*non-breaking space*): un carácter que se dibuja igual que
un espacio normal y que no lo es. Resultado: una categoría partida en dos, con 60 y 3 registros.

**`str.strip()` no lo arregla**, porque el carácter no está en los extremos sino en la mitad. Hay que
reemplazarlo explícitamente.

**Por qué este es el peor de los tres.** Los dos primeros producen un `KeyError` que detiene el
cuaderno y le señala el problema. Este produce un gráfico. Un gráfico con dos barras que se llaman
igual, que usted va a mirar sin entender por qué, o —peor— que va a poner en la sustentación sin
mirar. Es el mismo patrón que va a ver hoy tres veces: **el error que revienta se arregla; el error
que se ve bien se publica.**

**Pregunta de interpretación 1.** ¿Por qué un error que no lanza ninguna excepción es más peligroso
que uno que sí? Responda con lo que acaba de ver, no en abstracto.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Porque el que revienta le señala el problema y este no señala nada. Los errores 1 y 2 produjeron un
`KeyError` con el nombre de la columna adentro: el cuaderno se detiene, usted mira el mensaje y en
treinta segundos sabe qué escribir. El error 3 no produce ningún mensaje. Produce **un gráfico**: uno
con dos barras que se llaman `RENTAS PARAFISCALES`, una con 60 registros y otra con 3.

Y ahí está lo caro. Nadie publica un error que revienta, porque no hay nada que publicar. Un error
callado sí se publica: la figura sale, se ve bien, entra en la sustentación y nadie la mira dos
veces. La categoría partida en dos hace que un sector aparezca más pequeño de lo que es, y esa
diferencia se convierte en una conclusión falsa sin que nada haya fallado técnicamente.

Es el patrón que se repite hoy tres veces, y vale la pena aprendérselo con estas palabras: **el
error que revienta se arregla; el error que se ve bien se publica.**

</details>

---

## 2. La limpieza mínima

**El concepto.** Hoy no es una clase de limpieza: la limpieza fue la clase 3. Se limpia **solo lo que
impide graficar bien**, y son dos cosas.

1. **El espacio no separable**, que parte categorías en dos.
2. **Tres filas con `Apropiación Vigente` en cero.** Al calcular el porcentaje de ejecución esas
   filas dividen por cero.

**Sobre lo que NO hay que limpiar.** `Nombre Sector` y `Nombre Entidad` están limpios: 32 y 221
valores distintos, sin duplicados por espacios ni por mayúsculas. Parte del oficio es saber dónde
**no** buscar problemas: limpiar lo que ya está limpio es tiempo que el reto no le va a devolver.

**Los comandos.**

```python
serie.str.replace('\xa0', ' ', regex=False)   # reemplaza el caracter invisible
serie.str.strip()                             # quita espacios de los extremos
df[df['columna'] > 0].copy()                  # se queda con las filas utiles
```

Primero, qué pasa si no se filtran las filas en cero. En pandas una división por cero **no lanza
excepción**: produce `NaN` cuando el numerador también es cero, e `inf` cuando no lo es. En los dos
casos el cuaderno sigue corriendo como si nada. Otro error silencioso.

In [ ]:
ceros = (df['Apropiación Vigente'] == 0).sum()
print('Filas con apropiación en cero:', ceros)

# Qué produce dividir por cero, sin filtrar nada
ensayo = df['Compromisos'] / df['Apropiación Vigente'] * 100
print('Valores infinitos que aparecen:', np.isinf(ensayo).sum())
print('Valores NaN que aparecen:      ', ensayo.isna().sum())
print()
print('Ninguna de las dos líneas anteriores lanzó un error. Esos tres valores entran')
print('callados y después le rompen el promedio o el eje del gráfico, sin decirle por qué.')

In [ ]:
df_limpio = df.copy()

# 1. El espacio no separable, en todas las columnas de texto
for col in df_limpio.columns:
    if pd.api.types.is_string_dtype(df_limpio[col]):
        df_limpio[col] = df_limpio[col].str.replace('\xa0', ' ', regex=False).str.strip()

print('Categorías en Recursos´Presupuestales')
print('  antes: ', df[nombre_real].nunique())
print('  después:', df_limpio[nombre_real].nunique())

# 2. Las filas sin apropiación. Se filtran y se dice cuántas: nunca en silencio.
excluidas = (df_limpio['Apropiación Vigente'] == 0).sum()
df_limpio = df_limpio[df_limpio['Apropiación Vigente'] > 0].copy()

print()
print(f'Filas excluidas por apropiación en cero: {excluidas}')
print(f'Filas de trabajo: {len(df_limpio)}')
print(f'Sectores: {df_limpio["Nombre Sector"].nunique()} | '
      f'Entidades: {df_limpio["Nombre Entidad"].nunique()}')

**Pregunta de interpretación 2.** Las tres filas con apropiación en cero se podían haber filtrado
sin decir nada: el resultado numérico habría sido idéntico. ¿Por qué el cuaderno imprime cuántas se
fueron? Y una segunda parte: de las 21 columnas del archivo, ¿por qué **no** se limpiaron
`Nombre Sector` ni `Nombre Entidad`?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Por dos razones distintas, y las dos importan en la sustentación.

**Por qué se dice cuántas se fueron.** Un filtro es una decisión analítica, no un paso técnico. Quien
lea el análisis tiene derecho a saber que se excluyeron tres filas y por qué, porque de eso depende
si el resultado se puede generalizar. Tres de 3.645 no cambia nada; si hubieran sido 900, la
conclusión del día sería otra, y con el filtro silencioso nadie se habría enterado. La regla del
curso es corta: **se filtra en voz alta, nunca en silencio.** En la rúbrica esto vive en la dimensión
Ser, no en Hacer.

**Por qué no se limpiaron los nombres de sector y entidad.** Porque ya estaban limpios: 32 y 221
valores distintos, sin duplicados por espacios ni por mayúsculas, y eso se verificó antes de decidir.
Parte del oficio es saber dónde **no** buscar problemas. Limpiar lo que ya está limpio consume tiempo
del reto y, peor, introduce el riesgo de romper algo que funcionaba.

</details>

---

## 3. Para entender qué está pasando · Por qué un gráfico miente

Un gráfico no miente porque los números estén mal. En todo lo que sigue hoy **los números están
bien**: los tres gráficos malos y los tres rediseños usan exactamente las mismas cifras. La mentira
está en la traducción de número a forma, y ocurre por tres vías.

**1. El tipo de gráfico no responde la pregunta.** Cada tipo de gráfico codifica el valor en una
propiedad visual distinta: la barra en longitud, la línea en pendiente, el punto en posición, el pie
en ángulo. El ojo humano compara longitudes bien y ángulos mal. Elegir el tipo es elegir qué tan
difícil le queda la lectura al lector. Un pie con 32 porciones no está mal calculado: está mal
codificado.

**2. La escala distorsiona la comparación.** La longitud de una barra dice "esto es tanto". Si el eje
no arranca en cero, la longitud deja de ser proporcional al valor y la barra dice un número distinto
del que tiene. Ocurre lo mismo, en otra forma, con la escala logarítmica: comprime las diferencias
grandes. No está prohibida, pero **si se usa, se declara**.

**3. El gráfico afirma algo que los datos no contienen.** Una línea entre dos puntos afirma que hay
continuidad entre ellos. Si entre los dos puntos no hay nada —porque son categorías, no instantes—,
la línea es una afirmación falsa dibujada. Este es el peor de los tres, porque el gráfico se lee
perfectamente. El lector entiende. Entiende algo que no existe.

**La jerarquía que se lleva de hoy:**

> Un gráfico que se lee mal se descarta: el lector nota que no entiende. Un gráfico que se lee bien y
> miente, se cree.

### El data-ink ratio, sin matemáticas

Edward Tufte convirtió el diseño de gráficos en una disciplina con criterios verificables en lugar de
gustos. Su idea central cabe en una pregunta: **de toda la tinta del gráfico, ¿qué proporción
representa datos?** Todo lo demás es candidato a borrarse: el borde del recuadro, la caja de la
leyenda, la cuadrícula gruesa, los degradados, las sombras, el efecto 3D, el fondo de color.

La regla operativa: **borre. Si borra algo y el gráfico sigue diciendo lo mismo, ese algo sobraba.
Repita hasta que borrar duela.**

Y la advertencia contra el fanatismo, que importa tanto como la regla: una línea de referencia que
marca el promedio nacional es tinta que no representa datos, y es lo más útil de la figura. La
pregunta nunca fue "¿es tinta de datos?". Fue **"¿ayuda a leer?"**.

### 3.1 Las magnitudes: por qué el eje dice `1e13`

**El concepto.** `Apropiación Vigente` está en pesos, y el máximo por sector está alrededor de 97
billones. Cuando los números son enormes, matplotlib no escribe el número completo en el eje: escribe
`1e13` en una esquina y deja los ticks en 0, 2, 4, 6. El lector tiene que multiplicar mentalmente.
Casi nadie lo hace.

**El arreglo es una línea:** dividir entre `1e12` y decir "billones de pesos" en la etiqueta del eje.

Ese cambio de una línea es el atasco número uno del reto. La celda de abajo lo muestra con los dos
ejes lado a lado.

In [ ]:
apropiacion_por_sector = df_limpio.groupby('Nombre Sector')['Apropiación Vigente'].sum()
cinco = apropiacion_por_sector.sort_values(ascending=False).head(5)

fig, ejes = plt.subplots(1, 2, figsize=(14, 4))

ejes[0].barh(range(5), cinco.values, color='#B8C4CE')
ejes[0].set_yticks(range(5))
ejes[0].set_yticklabels(['' for _ in range(5)])
ejes[0].set_title('Pesos crudos: el eje dice 1e13')

ejes[1].barh(range(5), cinco.values / 1e12, color='#1F4E79')
ejes[1].set_yticks(range(5))
ejes[1].set_yticklabels(['' for _ in range(5)])
ejes[1].set_xlabel('Billones de pesos')
ejes[1].set_title('Dividido entre 1e12: el eje se lee')

plt.tight_layout()
plt.show()

**Pregunta de interpretación 3.** Los dos paneles tienen exactamente los mismos cinco valores y las
mismas cinco barras. No se corrigió ningún dato. ¿Qué cambió entonces, y por qué esa diferencia
cuenta como un problema de **veracidad** y no de estética?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Cambió quién hace la traducción. En el panel izquierdo el eje dice `0, 2, 4, 6` y arriba, en una
esquina diminuta, dice `1e13`. El número que el lector necesita —97 billones de pesos— no está
escrito en ninguna parte de la figura: hay que multiplicar mentalmente por diez billones. En el panel
derecho el número está dicho: la barra mide 97 y la etiqueta dice "billones de pesos".

No es estética porque el resultado no es "se ve más bonito", es **que casi nadie hace la
multiplicación**. Quien mira el panel izquierdo se lleva "seis coma algo" como orden de magnitud del
presupuesto de un sector, y eso es falso por siete órdenes de magnitud. Una figura que se lee mal
produce una lectura equivocada, no una lectura incómoda.

Y hay un detalle que conviene notar, porque es el atasco número uno del reto: **el arreglo es una
línea de código**, dividir entre `1e12`, más una etiqueta de eje que diga la unidad. Lo caro no es
arreglarlo, es darse cuenta.

</details>

---

## 4. La paleta del día y la función que limpia los ejes

**El concepto.** Una paleta es un **acuerdo**, no una decoración. Se define una vez, antes del primer
gráfico, y no se cambia. Cuatro roles bastan:

| Rol | Color | Cuándo se usa |
|-----|-------|---------------|
| Neutro | `#B8C4CE` | Todo por defecto. El gris es su amigo |
| Énfasis | `#1F4E79` | El único elemento que importa. Uno |
| Alerta | `#C0392B` | Solo cuando algo está bajo un umbral crítico |
| Referencia | `#7F8C8D` | Líneas de meta, anotaciones, la fuente |
| Secundario | `#E67E22` | La segunda serie, cuando de verdad hacen falta dos |

**Por qué el gris por defecto.** Porque el color es el canal que el lector obedece sin pensar: mira
primero lo que está coloreado. Si todo está coloreado, el lector elige solo, y usted perdió la
decisión más barata que tenía. La frase del bloque 1: *el gris no es el color de lo aburrido; el gris
es lo que hace que el azul signifique algo.*

**Azul contra gris, y azul contra naranja cuando hacen falta dos series. Nunca rojo contra verde**
como única distinción: es la combinación más común y la peor para daltonismo, que afecta a alrededor
del 8% de los hombres.

**Sobre `limpiar_ejes`.** Es la traducción a código del data-ink ratio, y la va a usar en todas las
figuras de hoy y del reto:

- `ax.spines[lado].set_visible(False)` borra los bordes del recuadro. Es la tinta que en el bloque 1
  todos señalaron como sobrante.
- `ax.set_axisbelow(True)` manda la grilla **detrás** de los datos. Sin eso la grilla se dibuja encima
  de las barras y las ensucia.
- La grilla queda en un solo eje, punteada y tenue. Es tinta que no representa datos y se queda,
  porque ayuda a leer.

In [ ]:
PALETA = {
    'neutro': '#B8C4CE',
    'enfasis': '#1F4E79',
    'alerta': '#C0392B',
    'referencia': '#7F8C8D',
    'secundario': '#E67E22',
}

FUENTE = 'Fuente: datos.gov.co - Ejecución del Presupuesto General de la Nación'

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = False


def limpiar_ejes(ax, eje_grilla='x'):
    """Quita los bordes sobrantes y deja una grilla tenue en un solo eje."""
    for lado in ['top', 'right', 'left']:
        ax.spines[lado].set_visible(False)
    if eje_grilla == 'x':
        ax.xaxis.grid(True, linestyle='--', alpha=0.3)
        ax.yaxis.grid(False)
    else:
        ax.yaxis.grid(True, linestyle='--', alpha=0.3)
        ax.xaxis.grid(False)
    ax.set_axisbelow(True)
    return ax


print('Paleta y estilo definidos.')

### 4.1 Los agregados que se van a graficar

**El concepto.** `groupby().sum()` es la analogía de los M&Ms de la clase 4: se separa por color y se
suma cada montón. Aquí se separa por sector, y por entidad, y se suma la plata.

Dos columnas derivadas que se calculan una vez y se usan todo el día:

- `% Ejecución` = compromisos sobre apropiación, en porcentaje. Es un **cociente**: no depende del
  tamaño del sector.
- `Apropiación (billones)` = la misma cifra dividida entre `1e12`. Es lo que va a los ejes.

Y **`sort_values` no es cosmética**: ordenar es lo que convierte una lista en un ranking. Sin orden,
el lector compara barra por barra; con orden, lee de arriba abajo y termina.

In [ ]:
columnas_plata = ['Apropiación Vigente', 'Compromisos', 'Obligaciones', 'Pagos']

sector = df_limpio.groupby('Nombre Sector')[columnas_plata].sum()
sector['% Ejecución'] = sector['Compromisos'] / sector['Apropiación Vigente'] * 100
sector['Apropiación (billones)'] = sector['Apropiación Vigente'] / 1e12
sector['Compromisos (billones)'] = sector['Compromisos'] / 1e12
sector = sector.sort_values('Apropiación Vigente', ascending=False)

entidad = df_limpio.groupby('Nombre Entidad')[['Apropiación Vigente', 'Compromisos']].sum()
entidad['% Ejecución'] = entidad['Compromisos'] / entidad['Apropiación Vigente'] * 100
entidad['Apropiación (billones)'] = entidad['Apropiación Vigente'] / 1e12
entidad = entidad.sort_values('Apropiación Vigente', ascending=False)

# La referencia nacional: el porcentaje de ejecución del país entero.
# Se calcula sobre las sumas, no como promedio de porcentajes.
ejecucion_nacional = (df_limpio['Compromisos'].sum() /
                      df_limpio['Apropiación Vigente'].sum() * 100)

print(f'Sectores: {len(sector)} | Entidades: {len(entidad)}')
print(f'Ejecución nacional: {ejecucion_nacional:.1f}%')
sector.head(10)[['Apropiación (billones)', 'Compromisos (billones)', '% Ejecución']].round(2)

### La respuesta que el pie chart no va a dejar leer

`sector` ya viene ordenada de mayor a menor apropiación, así que el tercer sector es el tercer
elemento del índice. La celda de abajo lo deja calculado **antes** de ver el gráfico de la sección
siguiente, a propósito: la respuesta está en los datos desde el principio; lo que el gráfico decide
es si el lector puede llegar a ella.

In [ ]:
tercer_sector = sector.index[2]

print('Tercer sector por apropiación vigente:', tercer_sector)
print('Los tres primeros:', list(sector.index[:3]))

**Pregunta de interpretación 4.** `ejecucion_nacional` se calculó dividiendo la **suma** de
compromisos entre la **suma** de apropiaciones, y no promediando los porcentajes de los 32 sectores.
Las dos cuentas dan números distintos. ¿Qué pregunta responde cada una, y por qué la del promedio de
porcentajes sería engañosa aquí?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

La cuenta que está en el código responde **"¿cuánto del presupuesto del país está comprometido?"**.
Es un cociente sobre el total, así que cada peso pesa lo mismo que cualquier otro peso.

Promediar los 32 porcentajes respondería otra cosa: **"¿cuál es el porcentaje del sector
promedio?"**. Ahí cada *sector* pesa lo mismo, no cada peso. Y en este archivo esa diferencia es
brutal, porque los sectores no se parecen en tamaño: el mayor tiene alrededor de 97 billones y el
menor está por debajo de uno. Un sector diminuto que ejecutó el 90% movería el promedio tanto como el
servicio de la deuda, que es lo que de verdad decide la cifra del país.

La regla general, y sirve para todo el semestre: **el promedio de unos cocientes no es el cociente de
los totales.** Cuál de los dos se usa depende de la pregunta, y esa es una decisión del analista que
hay que poder defender en voz alta. Esta línea de referencia va a aparecer dibujada en el rediseño 3;
si estuviera mal calculada, todos los colores de esa figura estarían mal asignados.

</details>

---

## 5. Rediseño 1 — El pie chart de 32 sectores

**La pregunta que el gráfico debería responder:**

> ¿Qué sectores concentran el presupuesto?

Mire el gráfico de abajo y, **antes de seguir leyendo**, intente responder con él:

**¿Cuál es el tercer sector con más presupuesto?**

Cuente cuánto tarda.

In [ ]:
# EL GRÁFICO MALO. No copie este código para nada real.
fig, ax = plt.subplots(figsize=(9, 9))

ax.pie(
    sector['Apropiación Vigente'],
    labels=sector.index,
    autopct='%1.2f%%',
    startangle=90,
    textprops={'fontsize': 7},
    wedgeprops={'edgecolor': 'white', 'linewidth': 0.5},
)
ax.set_title('Apropiación por sector', fontsize=16, fontweight='bold')
ax.legend(sector.index, loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=6)

plt.tight_layout()
plt.show()

### Diagnóstico

**Pregunta de interpretación 5.** ¿Pudo identificar el tercer sector mirando el pie? ¿Cuánto tardó?
Compárelo con el nombre que imprimió la celda de `tercer_sector`, y diga qué le impidió leerlo: el
tamaño de las porciones, la leyenda, los colores, o las tres cosas.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Casi nadie puede, y quien dice que sí normalmente leyó los porcentajes escritos encima de las
porciones, que es leer una tabla mal dispuesta, no leer un gráfico.

Los tres obstáculos son reales y son distintos:

- **El ángulo.** El pie codifica el valor en un ángulo, y el ojo humano estima ángulos mal. Dos
  porciones de 9,1% y 8,4% son indistinguibles. No es falta de práctica: es una limitación de la
  percepción, y por eso la crítica al pie no es una postura estética.
- **La leyenda de 32 entradas.** Obliga al ojo a saltar del gráfico a la leyenda y de vuelta, una vez
  por categoría. Ese ir y venir es trabajo que la figura le pasó al lector.
- **El arcoíris.** Con 32 colores automáticos, el color no significa nada, y el canal más obediente
  del ojo queda gastado.

Lo que hay que llevarse: **la respuesta estaba en los datos todo el tiempo.** El gráfico no perdió
información, la escondió. Y esa es la definición operativa de un gráfico mal elegido.

</details>

### Qué está mal, punto por punto

| Problema | Principio |
|----------|-----------|
| Tipo de gráfico equivocado para comparar magnitudes | El ojo compara longitudes bien y ángulos mal |
| 32 categorías en un pie | Un pie funciona con 2 o 3 porciones, y solo si una domina |
| Arcoíris sin significado | El color se reserva para el mensaje |
| Leyenda de 32 entradas | Obliga al ojo a saltar ida y vuelta entre leyenda y gráfico |
| Porcentajes con dos decimales | Finge una precisión que el gráfico no permite leer |
| Título que describe los ejes | El título debe decir la conclusión |

**Por qué el pie falla, en una frase que no es una opinión.** El pie codifica el valor en un ángulo, y
el ojo humano estima ángulos mal. Tres porciones de 35%, 33% y 32% son indistinguibles en un pie y
triviales en barras. No es una postura estética: es una limitación de la percepción.

**Y el criterio para las excepciones, porque las hay:** un pie funciona con dos o tres porciones,
cuando una domina claramente y el mensaje es "esto es la mayoría". Fuera de ahí, barras.

### El rediseño

Barras horizontales, top 10, ordenadas, todo gris salvo la primera, valores etiquetados directamente
al final de cada barra, sin leyenda, sin bordes, título con la conclusión.

Ninguna de esas decisiones es de gusto. Cada una sale de un punto de la checklist.

In [ ]:
# EL GRÁFICO BUENO
top10 = sector.head(10).sort_values('Apropiación (billones)')

fig, ax = plt.subplots(figsize=(10, 6))

# La técnica del gris más resaltado: todo neutro, uno en énfasis.
# barh dibuja de abajo hacia arriba, así que el mayor es el ÚLTIMO de la lista.
colores = [PALETA['neutro']] * len(top10)
colores[-1] = PALETA['enfasis']

ax.barh(top10.index, top10['Apropiación (billones)'], color=colores)

# Etiquetas directas al final de cada barra
for i, valor in enumerate(top10['Apropiación (billones)']):
    ax.text(valor + 1.5, i, f'{valor:,.1f}', va='center', fontsize=10, color='#333333')

ax.set_xlim(0, top10['Apropiación (billones)'].max() * 1.15)
ax.set_xlabel('Apropiación vigente (billones de pesos)')
ax.set_title('La deuda pública pesa más que cualquier sector social del presupuesto',
             fontsize=13, fontweight='bold', loc='left', pad=15)
limpiar_ejes(ax, eje_grilla='x')
ax.tick_params(axis='y', length=0, labelsize=9)

fig.text(0.01, -0.02, FUENTE, fontsize=8, color=PALETA['referencia'])
plt.tight_layout()
plt.show()

### Para entender qué está pasando · Qué hace cada decisión

**`sector.head(10)`: recortar es una decisión analítica.** Los 22 sectores restantes suman poco y
solo agregan ruido. Recortar está bien; lo que no está bien es recortar sin poder defenderlo. Si
alguien pregunta "¿y los otros 22?", la respuesta tiene que existir.

**`sort_values`: ordenar es lo que crea el ranking.** Y el detalle traicionero: `barh` dibuja de abajo
hacia arriba, así que hay que ordenar **ascendente** para que la barra más grande quede arriba.

**`barh` en lugar de `bar`.** Mire el nombre "INCLUSION SOCIAL Y RECONCILIACION". En vertical ese
texto se rota 90 grados y se vuelve ilegible. **Nombres largos piden barras horizontales.** La
solución al texto ilegible nunca es rotarlo más.

**La lista de colores.** `color=` acepta un color, o una lista con un color por barra. Esa lista es
toda la técnica del gris más resaltado, y es el 80% del efecto visual con el 5% del esfuerzo. La va a
usar el resto del semestre.

**`ax.text()`: etiquetar directamente.** Con el valor escrito al final de la barra, el lector ya no
necesita seguir la barra hasta el eje y volver. Es data-ink ratio aplicado: se agrega tinta que sí
representa datos y se ahorra el trabajo de lectura.

**`ax.set_xlim(0, ...)` con un 15% de holgura.** El cero es obligatorio (sección 6), y la holgura
evita que la etiqueta de la barra más larga se salga de la figura.

**El título.** "Apropiación por sector" describe los ejes, que ya están escritos abajo. El renglón más
leído de la figura se usa para decir lo que usted encontró.

**Pregunta de interpretación 6.** Los dos gráficos tienen exactamente los mismos números. De los
tres cambios —el tipo de gráfico, el orden y el color—, ¿cuál hizo más por la legibilidad? Defienda
su elección diciendo qué se perdería si se deshace solo ese cambio y se dejan los otros dos.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

La respuesta defendible es **el tipo de gráfico**, y la prueba es deshacer cada cambio por separado:

- Si vuelve al **pie** y conserva el orden y el gris más resaltado, sigue sin poderse leer: las
  porciones ordenadas y grises siguen siendo ángulos. El cambio de barra por porción es el que
  devuelve la comparación al canal que el ojo mide bien, la longitud.
- Si conserva las barras y quita el **orden**, todavía se puede leer cada barra, pero hay que
  recorrerlas una por una para armar el ranking. Se pierde velocidad, no información.
- Si conserva las barras ordenadas y quita el **color** —todas iguales—, la figura sigue siendo
  correcta. Se pierde el énfasis, que es lo que dirige la mirada al hallazgo.

O sea: el tipo de gráfico decide si **se puede** leer, el orden decide qué tan **rápido**, y el color
decide **qué** se mira primero. Esa jerarquía es el orden en que conviene atacar una figura propia en
el reto, y explica por qué el punto 2 de la checklist va antes que el 3 y que el 6.

Y una observación sobre el título: "Apropiación por sector" describe los ejes, que ya están escritos
abajo. El renglón más leído de la figura se usa para decir lo que usted encontró.

</details>

---

## 6. Rediseño 2 — Barras arcoíris con eje truncado

**La pregunta:**

> ¿Qué entidades concentran el presupuesto?

Mire el gráfico y responda antes de seguir:

**¿Cuántas veces más grande es la barra más alta que la más baja?**

In [ ]:
# EL GRÁFICO MALO. Este es el más deshonesto de los tres.
top_entidades = entidad.head(10)
top_alfabetico = top_entidades.sort_index()          # orden alfabético, a propósito

fig, ax = plt.subplots(figsize=(11, 6))

arcoiris = plt.cm.rainbow(np.linspace(0, 1, len(top_alfabetico)))
ax.bar(range(len(top_alfabetico)), top_alfabetico['Apropiación (billones)'],
       color=arcoiris, edgecolor='black', linewidth=1.2)

# EL TRUNCAMIENTO: el eje arranca en 8, no en 0
ax.set_ylim(8, top_alfabetico['Apropiación (billones)'].max() * 1.02)

ax.set_xticks(range(len(top_alfabetico)))
ax.set_xticklabels(top_alfabetico.index, rotation=90, fontsize=7)
ax.set_title('Ejecución presupuestal', fontsize=16, fontweight='bold')
ax.grid(True, linewidth=1.5, color='gray')
for lado in ['top', 'right']:
    ax.spines[lado].set_visible(True)

plt.tight_layout()
plt.show()

### Medir la mentira, en vez de discutirla

Aquí está la diferencia entre **discutir** el eje truncado y **medirlo**. Con el eje en cero, la
altura dibujada de una barra es su valor. Con el eje arrancando en 8, la altura dibujada es
`valor - 8`. La razón entre las dos alturas dibujadas es lo que el lector percibe, y no se parece a
la razón entre los valores reales.

In [ ]:
valores = top_entidades['Apropiación (billones)']

razon_real = valores.max() / valores.min()
razon_aparente = round((valores.max() - 8) / (valores.min() - 8), 2)

print(f'Razón real:     {razon_real:.2f} veces')
print(f'Razón aparente: {razon_aparente:.2f} veces')

**Pregunta de interpretación 7.** Compare el número que usted estimó a ojo mirando el gráfico con
las dos razones que acaba de imprimir. ¿Qué historia contaba el gráfico truncado que no era cierta?
Y una segunda parte, más incómoda: nadie falsificó un dato. ¿Dónde está exactamente la mentira?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

La razón real está alrededor de **1,7 veces**: las diez entidades más grandes son grandes todas, y
entre la primera y la última hay menos diferencia de la que uno esperaría. La razón aparente pasa de
**10 veces**. El gráfico contaba que una entidad concentra un orden de magnitud más presupuesto que
otra, y eso no es cierto ni de lejos.

Sobre dónde está la mentira: no está en ningún número, está en la **traducción de número a forma**.
Una barra codifica magnitud mediante longitud, y el lector no lee la cifra, lee el largo. Al arrancar
el eje en 8, el largo deja de ser proporcional al valor: la barra de la entidad menor no mide 15,4,
mide 7,4, y ese 7,4 no corresponde a ningún dato del archivo. **La figura afirma algo falso por
construcción**, sin que nadie tenga que tocar el CSV.

Por eso el eje truncado no es un asunto de estilo y la regla es dura: en barras, el eje arranca en
cero. Siempre.

</details>

### Para entender qué está pasando · Por qué truncar barras es mentir

Una barra codifica magnitud mediante **longitud**. El lector no lee el número: lee el largo. Si el eje
no arranca en cero, la longitud deja de ser proporcional al valor, y entonces el largo que el lector
lee no corresponde a ningún número del dataset. La barra afirma algo falso **por construcción**, sin
que nadie tenga que falsificar un dato.

En un gráfico de líneas la respuesta es distinta, y esa diferencia es el tradeoff 1 del bloque 1. La
línea no codifica magnitud, codifica **cambio**: la información está en la pendiente. Si la variación
relevante es pequeña frente al nivel —una tasa que se mueve entre 88% y 92%—, obligar al eje a
arrancar en cero aplasta la señal y esconde algo real. Ahí truncar es defendible, **y se declara
visiblemente en el eje**.

La pregunta que decide, y sirve para cualquier caso:

> ¿El lector está comparando magnitudes, o siguiendo un cambio?

Magnitudes: cero obligatorio. Cambio: se puede truncar, y se dice.

**Truncar en silencio es lo que se llama manipular.** No es un adjetivo moral suelto: es la diferencia
entre una decisión de diseño y un dato falso.

### El resto de los problemas del gráfico malo

| Problema | Principio |
|----------|-----------|
| Eje truncado | La longitud de la barra debe ser proporcional al valor |
| Orden alfabético | Ordenar por valor es lo que crea el ranking |
| Arcoíris | Diez colores, cero información |
| Etiquetas rotadas 90 grados | Nombres largos piden barras horizontales |
| Grilla gruesa y gris | Data-ink ratio: compite con los datos |
| Bordes negros en las barras | Tinta que no aporta nada |
| Título genérico | "Ejecución presupuestal" no dice nada |

### El rediseño de las entidades

Es la técnica del rediseño 1 aplicada a otra tabla, y viene escrita entera porque **es el modelo que
el reto sí le va a cobrar**: dibujada, ordenada, con el eje desde cero, con las dos etiquetas de eje,
con las etiquetas de valor sobre las barras y con un título que dice una conclusión verdadera.

Léala línea por línea antes de ejecutarla. En el reto va a escribir cinco figuras como esta sin
tenerla al lado, y lo que se copia sin leer se nota, porque los nombres de las columnas cambian.

In [ ]:
datos = entidad.head(10).sort_values('Apropiación (billones)')

fig, eje_entidades = plt.subplots(figsize=(10, 6))

colores = [PALETA['neutro']] * len(datos)
colores[-1] = PALETA['enfasis']

eje_entidades.barh(datos.index, datos['Apropiación (billones)'], color=colores)

for i, valor in enumerate(datos['Apropiación (billones)']):
    eje_entidades.text(valor + 0.8, i, f'{valor:,.1f}', va='center', fontsize=10,
                       color='#333333')

eje_entidades.set_xlim(0, datos['Apropiación (billones)'].max() * 1.15)
eje_entidades.set_xlabel('Apropiación vigente (billones de pesos)')
eje_entidades.set_ylabel('Entidad')
eje_entidades.set_title('El servicio de la deuda concentra más presupuesto que cualquier ministerio',
                        fontsize=13, fontweight='bold', loc='left', pad=15)
limpiar_ejes(eje_entidades, eje_grilla='x')
eje_entidades.tick_params(axis='y', length=0, labelsize=8)

fig.text(0.01, -0.02, FUENTE, fontsize=8, color=PALETA['referencia'])
plt.tight_layout()
plt.show()

**Pregunta de interpretación 8.** En barras el cero es obligatorio, y acaba de ver por qué. Pero el
tradeoff 1 del bloque 1 decía que en un gráfico de **líneas** truncar el eje puede ser legítimo.
¿Por qué la misma operación es mentira en un caso y defendible en el otro? ¿Y qué hay que hacer
siempre que se trunque?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Porque las dos formas codifican cosas distintas.

- **La barra codifica magnitud, mediante longitud.** El mensaje es "esto es tanto". Si el origen se
  mueve, la longitud deja de ser proporcional al valor y el mensaje se vuelve falso. No hay caso en
  que se salve.
- **La línea codifica cambio, mediante pendiente.** El mensaje es "esto se movió así". La información
  está en la inclinación, no en la distancia al eje. Si la variación relevante es pequeña frente al
  nivel —una tasa que se mueve entre 88% y 92%—, forzar el cero aplasta la señal y esconde algo real.

La pregunta que decide, y sirve para cualquier figura: **¿el lector está comparando magnitudes o
siguiendo un cambio?** Magnitudes, cero obligatorio. Cambio, se puede truncar.

Y lo que hay que hacer siempre: **declararlo visiblemente en el eje.** Truncar y decirlo es una
decisión de diseño; truncar en silencio es manipular. Esa es la diferencia entera, y cabe en una
línea de código y una etiqueta.

</details>

---

## 7. Rediseño 3 — Líneas sobre categorías

**La pregunta:**

> ¿Qué sectores ejecutan más y cuáles menos?

Este es el más peligroso de los tres, y por una razón incómoda: **se ve bien.**

In [ ]:
doce_sectores = sector.head(12)
doce = doce_sectores.sort_index()          # orden alfabético

# EL GRÁFICO MALO. Se ve razonablemente bien y afirma algo falso.
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(doce.index, doce['% Ejecución'], marker='o', linewidth=2,
        color='#E74C3C', markersize=8)
ax.set_xticks(range(len(doce)))
ax.set_xticklabels(doce.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('% Ejecución')
ax.set_title('Evolución de la ejecución presupuestal', fontsize=15, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Diagnóstico

**Pregunta de interpretación 9.** ¿Qué significa la pendiente entre dos sectores consecutivos, por
ejemplo entre "CULTURA" y "DEFENSA Y POLICIA"? Antes de responder, hágase esta prueba mental: si
ordena los sectores al revés, ¿qué le pasa a esa pendiente?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**No significa nada.** No hay absolutamente nada entre esos dos puntos: son dos categorías que
quedaron juntas porque el alfabeto las puso juntas.

Y la prueba mental lo demuestra sin discutir: si ordena los sectores al revés, **todas las pendientes
cambian de signo sin que cambie un solo dato**. Una propiedad visual que depende del orden en que
usted escribió las filas no está codificando información: la está inventando.

Una línea conecta puntos porque **afirma que existe continuidad entre ellos**. Entre enero y febrero
hay un continuo de tiempo, y por eso la pendiente significa "cambió a esta velocidad". Aquí no hay
ningún continuo, y conviene recordar el dato de la sección 0: **este archivo no tiene ninguna columna
de tiempo.** Es una foto. El título del gráfico malo dice "Evolución" y no hay ninguna evolución que
mostrar.

</details>

### Para entender qué está pasando · Qué afirma una línea

**No significa nada.** No hay absolutamente nada entre esos dos puntos.

Un gráfico de líneas conecta puntos porque **afirma que existe continuidad entre ellos**. Entre enero
y febrero hay un continuo de tiempo, y por eso la pendiente significa "cambió a esta velocidad".
Entre "CULTURA" y "DEFENSA Y POLICIA" no hay nada: son dos categorías que quedaron juntas porque el
alfabeto las puso juntas.

**La prueba definitiva:** si ordena los sectores al revés, todas las pendientes cambian de signo
**sin que cambie un solo dato**. Una propiedad visual que depende del orden en que usted escribió las
filas no está codificando información: la está inventando.

Y recuerde lo que se anunció en el bloque 1: **este dataset no tiene ninguna columna de tiempo.** Es
una foto. Cualquier línea que se dibuje hoy está contando una película que no existe. El título del
gráfico malo dice "Evolución", y no hay ninguna evolución que mostrar.

**Por qué este es el peor de los tres:**

| Gráfico | Problema | Consecuencia |
|---------|----------|--------------|
| Pie de 32 sectores | Ilegible | Se descarta. El lector nota que no entiende |
| Barras truncadas | Exagera | Engaña, pero un lector atento revisa el eje |
| Líneas sobre categorías | **Se lee bien y afirma algo falso** | El lector entiende. Entiende algo que no existe |

El error del pie es de legibilidad. El del eje truncado es de escala. **El de la línea es de
veracidad**, y por eso es el que la rúbrica del reto castiga con un tope de banda.

### El número que va a decidir el título

Un 13% de ejecución no significa nada por sí solo: ¿es bueno? ¿es malo? Solo significa algo contra un
estándar, y por eso el rediseño va a necesitar una **línea de referencia**.

Contar antes de graficar es lo que permite escribir un título que dice una conclusión **y que es
verdadero**. El orden importa: primero se cuenta, después se dibuja, y el título sale del conteo.

In [ ]:
sectores_bajo_promedio = int((doce_sectores['% Ejecución'] < ejecucion_nacional).sum())

print(f'{sectores_bajo_promedio} de 12 sectores ejecutan por debajo del '
      f'{ejecucion_nacional:.1f}% nacional')

### El rediseño de referencia

El gráfico correcto para "comparar un valor entre categorías" es de la familia de las barras. Aquí va
un **dot plot** (o *lollipop*): la misma información, menos tinta, y funciona bien cuando los valores
son porcentajes en un rango estrecho.

Tres decisiones nuevas, y las tres son de criterio:

1. **Ordenado por valor**, no alfabético. Con eso ya se puede leer un ranking.
2. **Línea de referencia** en la ejecución nacional. Es tinta que no representa datos y es lo más útil
   de la figura. La pregunta nunca fue "¿es tinta de datos?", fue "¿ayuda a leer?".
3. **Color condicional:** los sectores por debajo de la referencia van en color de alerta. Es la
   primera vez hoy que el color codifica un **hecho** y no un énfasis. Aquí el rojo sí se justifica,
   porque significa "por debajo de la meta", que es lo que el rojo significa culturalmente.

Y el respaldo obligatorio: **el color nunca es el único canal.** La posición en el ranking, la línea
de referencia y la etiqueta del valor dicen lo mismo que el color. Quien no distinga los colores
recibe el mensaje igual.

In [ ]:
datos = doce_sectores.sort_values('% Ejecución')

fig, ax = plt.subplots(figsize=(10, 7))

colores = [PALETA['alerta'] if v < ejecucion_nacional else PALETA['enfasis']
           for v in datos['% Ejecución']]

# hlines dibuja el "palo" del lollipop; scatter dibuja el "dulce" en la punta
ax.hlines(y=datos.index, xmin=0, xmax=datos['% Ejecución'],
          color=PALETA['neutro'], linewidth=2)
ax.scatter(datos['% Ejecución'], datos.index, color=colores, s=110, zorder=3)

ax.axvline(ejecucion_nacional, color=PALETA['referencia'], linestyle='--', linewidth=1.5)
ax.text(ejecucion_nacional + 0.7, -0.7, f'Promedio nacional: {ejecucion_nacional:.1f}%',
        fontsize=9, color=PALETA['referencia'])

for i, valor in enumerate(datos['% Ejecución']):
    ax.text(valor + 1.2, i, f'{valor:.1f}%', va='center', fontsize=9, color='#333333')

ax.set_xlim(0, datos['% Ejecución'].max() * 1.25)
ax.set_xlabel('Porcentaje de la apropiación ya comprometida')
ax.set_ylabel('Sector')

# El título sale del número que se contó arriba. Sin ese conteo, el título
# se quedaría en algo descriptivo, que es justo el que no queremos.
titulo = (f'{sectores_bajo_promedio} de los 12 sectores más grandes ejecutan '
          f'por debajo del promedio nacional')
ax.set_title(titulo, fontsize=13, fontweight='bold', loc='left', pad=15)
limpiar_ejes(ax, eje_grilla='x')
ax.tick_params(axis='y', length=0, labelsize=9)

fig.text(0.01, -0.02, FUENTE, fontsize=8, color=PALETA['referencia'])
plt.tight_layout()
plt.show()

### Para entender qué está pasando · Cómo se arma un lollipop

**No es un tipo de gráfico de matplotlib.** Se arma con dos capas: `hlines` dibuja el palo desde 0
hasta el valor, y `scatter` pone el punto en la punta. El `zorder=3` del scatter lo dibuja **encima**
de la línea; sin eso el palo se ve por encima del punto y queda sucio.

**Por qué lollipop y no barras.** Con barras, la mayor parte de la tinta está en el relleno, y aquí
los valores viven en un rango estrecho: las barras se ven casi iguales y la masa de tinta compite con
la lectura. El punto marca la posición exacta con mucha menos tinta. Es una decisión de data-ink
ratio y es defendible en los dos sentidos: **barras también habría sido correcto.** Lo que no era
correcto era la línea.

**`ax.axvline()`.** La línea de referencia. Es el ejemplo perfecto del matiz contra el minimalismo
dogmático: tinta que no representa datos, y se queda.

**El color condicional.** La lista por comprensión evalúa cada valor contra el umbral y devuelve un
color por punto. Es la misma técnica del gris más resaltado, con la condición cambiada.

**Pregunta de interpretación 10.** La línea punteada del promedio nacional es tinta que **no**
representa ningún dato del archivo: no es un sector, no es un valor de ninguna fila. Según el
data-ink ratio de Tufte tendría que borrarse, y sin embargo es lo más útil de la figura. ¿Cómo se
resuelve esa contradicción, y qué pregunta reemplaza a "¿es tinta de datos?"?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

No es una contradicción, es que la regla estaba mal enunciada. La pregunta de Tufte nunca fue **"¿es
tinta de datos?"**, fue **"¿ayuda a leer?"**. Borrar es el reflejo por defecto porque casi toda la
tinta que sobra —bordes, cajas de leyenda, cuadrículas gruesas, sombras, 3D— no ayuda a nadie. Pero el
criterio es la lectura, no la contabilidad de píxeles.

Y aquí la línea es lo que convierte un número en un juicio. Sin ella, "13,5%" es una cifra suelta y el
lector no tiene contra qué compararla; con ella, "13,5%" es *por debajo de lo que ejecuta el país*. La
línea es la que hace que los colores signifiquen algo, la que permite escribir el título del rediseño
y la que sostiene la conclusión.

La versión aplicable de la regla: **borre todo lo que no ayude a leer, y agregue lo que ayude aunque
no sea un dato.** El minimalismo dogmático produce figuras limpias que no dicen nada, y ese es un
error tan caro como la cuadrícula gruesa.

</details>

### La misma figura, otro recorte y otra pregunta

Entre los 20 sectores con más presupuesto, ¿cuáles son los 10 que **menos** ejecutan?

Es el mismo gráfico con un matiz que sí cambia: el recorte ya no es "los más grandes" sino "los que
peor ejecutan **entre** los grandes". Filtrar primero por tamaño y después por desempeño es una
decisión analítica, y hay que poder defenderla: sin el primer filtro, la lista se llenaría de
sectores diminutos cuyo porcentaje se mueve con nada.

Esta figura y la anterior son el modelo de las cinco del reto. Las dos traen título, las dos
etiquetas de eje, las etiquetas de valor y la línea de referencia.

In [ ]:
peores = sector.head(20).nsmallest(10, '% Ejecución').sort_values('% Ejecución')

fig, eje_lollipop = plt.subplots(figsize=(10, 6))

colores = [PALETA['alerta'] if v < ejecucion_nacional else PALETA['enfasis']
           for v in peores['% Ejecución']]

eje_lollipop.hlines(y=peores.index, xmin=0, xmax=peores['% Ejecución'],
                    color=PALETA['neutro'], linewidth=2)
eje_lollipop.scatter(peores['% Ejecución'], peores.index, color=colores, s=110, zorder=3)
eje_lollipop.axvline(ejecucion_nacional, color=PALETA['referencia'],
                     linestyle='--', linewidth=1.5)

for i, valor in enumerate(peores['% Ejecución']):
    eje_lollipop.text(valor + 0.6, i, f'{valor:.1f}%', va='center', fontsize=9,
                      color='#333333')

eje_lollipop.set_xlim(0, peores['% Ejecución'].max() * 1.35)
eje_lollipop.set_xlabel('Porcentaje de la apropiación ya comprometida')
eje_lollipop.set_ylabel('Sector')
eje_lollipop.set_title('Los diez sectores grandes que menos han comprometido su presupuesto',
                       fontsize=13, fontweight='bold', loc='left', pad=15)
limpiar_ejes(eje_lollipop, eje_grilla='x')
eje_lollipop.tick_params(axis='y', length=0, labelsize=9)

plt.tight_layout()
plt.show()

---

## 8. Cierre del bloque 2

### La checklist aplicada

Páseles la checklist a los tres rediseños. Los tres deben pasar los diez puntos.

| # | Criterio | Rediseño 1 | Rediseño 2 | Rediseño 3 |
|---|----------|-----------|-----------|-----------|
| 1 | ¿Pasa la prueba de los 5 segundos? | | | |
| 2 | ¿El tipo de gráfico corresponde a la pregunta? | | | |
| 3 | ¿Categorías ordenadas por valor? | | | |
| 4 | ¿El eje empieza en cero, o hay razón declarada? | | | |
| 5 | ¿Borré todo lo que se podía borrar? | | | |
| 6 | ¿Un solo elemento resaltado, el resto en gris? | | | |
| 7 | ¿Funciona en escala de grises? | | | |
| 8 | ¿El título dice la conclusión y es verdadero? | | | |
| 9 | ¿Los ejes tienen unidades? | | | |
| 10 | ¿Está la fuente del dato? | | | |

**Pregunta de interpretación 11.** Llene la tabla y diga si algún rediseño falla algún punto. El 7 es
el que más se escapa: mire las figuras entrecerrando los ojos, o imprímalas en blanco y negro
mentalmente.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Los tres pasan los diez, y conviene saber **por qué** pasan cada uno, porque es el argumento que hay
que dar en la sustentación:

- **Punto 4, el cero.** Los tres arrancan en cero, y en los tres es obligatorio: los tres comparan
  magnitudes, ninguno sigue un cambio.
- **Punto 6, un solo elemento resaltado.** Los rediseños 1 y 2 usan gris más énfasis. El rediseño 3
  es la excepción declarada: ahí el color no marca énfasis, marca un **hecho** —estar por debajo de la
  referencia—, y por eso hay más de un punto coloreado sin romper la regla.
- **Punto 7, la escala de grises.** Es el que más se escapa, y el rediseño 3 es el candidato a
  fallarlo, porque azul y rojo en gris quedan parecidos. **Pasa igual**, y esa es la lección: la
  posición en el ranking, la línea de referencia y la etiqueta del valor dicen lo mismo que el color.
  Quien no distinga los colores recibe el mensaje completo. **El color nunca es el único canal.**
- **Punto 8, el título verdadero.** Los tres títulos afirman algo comprobable contra la tabla, y el
  del rediseño 3 sale de un conteo que se hizo antes de dibujar. Ese es el único punto de la
  checklist que exige haber contado.

Si en su copia algún punto falla —por ejemplo porque cambió un recorte—, eso no es un error del
cuaderno: es exactamente el trabajo del reto.

</details>

**Pregunta de interpretación 12.** De los tres gráficos malos, ¿cuál era el peor y por qué?
Justifique con el criterio de la sección 7, no con lo feo que se veía.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

El de **líneas**, y casi todo el mundo dice el pie.

El pie es ilegible, y **eso se nota**. Un gráfico que se lee mal se descarta: el lector percibe que no
está entendiendo, hace perder tiempo y nada más. El de barras truncadas exagera, y engaña, pero deja
rastro: un lector atento mira el eje y detecta el truco.

El de líneas se lee perfectamente **y afirma algo falso**. El lector entiende. Entiende algo que no
existe. Y no tiene forma de sospecharlo desde la figura, porque no hay nada raro que mirar: no es un
eje sospechoso ni una leyenda imposible, es una continuidad inventada entre categorías.

La jerarquía que se lleva de hoy, y que la rúbrica del reto aplica con un tope de banda: **un gráfico
que se lee mal se descarta; un gráfico que se lee bien y miente, se cree.** El error del pie es de
legibilidad, el del eje truncado es de escala, y el de la línea es de **veracidad**.

</details>

### Preguntas que siempre salen

**¿Los pie charts son realmente tan malos?**
No son malos por naturaleza; están mal usados casi siempre. Tres porciones de 35%, 33% y 32% son
indistinguibles en un pie y triviales en barras. Un pie funciona con dos o tres porciones cuando una
domina claramente. Fuera de ahí, barras.

**Si los dos gráficos son correctos, ¿cuál elijo?**
El que reduce la fricción para la pregunta que quiere que el lector responda. "Correcto" es el
mínimo, no el criterio. El lollipop y las barras del rediseño 3 son los dos correctos; se eligió el
lollipop por el rango estrecho de los valores.

**¿Truncar un eje es siempre manipular?**
En barras, sí: la barra codifica magnitud con longitud. En líneas no necesariamente, porque la línea
codifica cambio. La regla es: si trunca, lo declara visiblemente. Truncar en silencio es manipular.

**¿Y si mi gráfico necesita más de cinco colores?**
Casi siempre significa que está metiendo demasiadas categorías en una figura. Opciones: agrupar las
menores en "otros", partir en dos figuras, o pasar a small multiples. Si de verdad necesita quince
colores, el lector tampoco va a distinguir quince colores.

**¿Un título que dice la conclusión no es sesgar al lector?**
Es dirigirlo, y esa es su responsabilidad: usted ya hizo el análisis, no lo obligue a repetirlo. La
línea que no se cruza es la verdad. Si el título afirma algo que los datos no sostienen, eso ya no es
dirigir, es mentir, y en el renglón más leído de la figura.

**Mi paleta se ve bien en mi pantalla. ¿Es suficiente?**
No. Tres pruebas: el proyector del salón, que aplasta el contraste; la impresión en escala de grises;
y un simulador de daltonismo. La tercera es la que más gente omite y la que afecta a personas reales.

**¿Puedo usar IA para hacer los gráficos del reto?**
Sí, y para eso fue la clase 7. Pero si le pide "un gráfico bonito" le va a dar algo genérico. Si le
pide "barras horizontales ordenadas, top 10, todas en gris salvo la primera, con etiquetas directas,
eje desde cero y sin bordes", le va a dar exactamente eso. **La diferencia entre las dos peticiones es
lo que se aprendió hoy.**

### Resumen

| Lo que hizo | Con qué |
|-------------|---------|
| Barras horizontales | `ax.barh(etiquetas, valores, color=lista_de_colores)` |
| Gris más resaltado | `colores = [neutro] * n` y después `colores[-1] = enfasis` |
| Etiqueta directa de valor | `ax.text(x, y, texto, va='center')` |
| Forzar el cero | `ax.set_xlim(0, maximo * 1.15)` |
| Borrar los bordes | `ax.spines[lado].set_visible(False)` |
| Grilla detrás de los datos | `ax.set_axisbelow(True)` |
| Lollipop | `ax.hlines(y=, xmin=0, xmax=)` más `ax.scatter(..., zorder=3)` |
| Línea de referencia | `ax.axvline(valor, linestyle='--')` |
| Color condicional | `[alerta if v < umbral else enfasis for v in serie]` |
| Título con conclusión | `ax.set_title('...', loc='left', pad=15)` |
| Unidades legibles | dividir entre `1e12` y decirlo en `set_xlabel` |
| La fuente al pie | `fig.text(0.01, -0.02, FUENTE, fontsize=8)` |

**Las siete reglas que no se negocian:**

1. El gráfico se deriva de la pregunta, no del gusto ni de lo que se sabe hacer.
2. En barras, el eje arranca en cero. Siempre.
3. Nunca líneas sobre categorías: la línea afirma una continuidad que no existe.
4. Ordenar por valor, salvo que exista un orden natural (estratos, meses, niveles).
5. Todo gris, y el color se gana. Un elemento resaltado, uno.
6. El color nunca es el único canal: posición, orden o etiqueta directa lo respaldan.
7. El título dice la conclusión, y la conclusión es verdadera.

**Autoevaluación honesta.** Si puede responder que sí a estas cinco, está listo para el reto:

- [ ] Puedo elegir el tipo de gráfico a partir de la pregunta, y decir por qué no elegí el vecino.
- [ ] Sé explicar, con un número, por qué truncar el eje de unas barras es una afirmación falsa.
- [ ] Puedo hacer barras horizontales ordenadas, en gris con una resaltada y con etiquetas directas,
      sin copiar de aquí.
- [ ] Sé cuándo una línea de referencia justifica su tinta.
- [ ] Puedo escribir un título que dice una conclusión y verificarlo contra la tabla.

**Siguiente:** bloque 3, el reto. Mismo dataset, cinco preguntas nuevas, cuatro tipos de gráfico que
no aparecieron aquí.